# Model Training

In this notebook we load the cleaned dataset, split it into training and test sets, train each of our six models, and look at how well each one did on the test data.

The goal is to predict the next day AQHI value for the Montreal Trudeau Airport region based on the current day pollutant readings.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import sklearn

from load_data import load_data
from models import linear_regression_model_base, linear_regression_model_lag, lasso_regression_model, decision_tree_regression_model, random_forest_regression_model
from features import TARGET_VARIABLE
from evaluate import get_param_grid_DT, get_best_DT_model, model_evaluation, get_param_grid_RF, get_best_RF_model, get_best_lasso_model, get_param_grid_lasso
from sklearn.model_selection import train_test_split

## Step 1: Load the Dataset

We load the processed dataset that was built from the NAPS air quality measurements for Montreal (2020 to 2024). Each row represents one day.

In [2]:
data = load_data()
print("Dataset shape:", data.shape)
print()
print(data.info())
data.head()

Dataset shape: (1771, 12)

<class 'pandas.DataFrame'>
RangeIndex: 1771 entries, 0 to 1770
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              1771 non-null   datetime64[us]
 1   Day               1771 non-null   int64         
 2   Month             1771 non-null   int64         
 3   CO                1771 non-null   float64       
 4   NO2               1771 non-null   float64       
 5   O3                1771 non-null   float64       
 6   PM25              1771 non-null   float64       
 7   AQHI_PreviousDay  1771 non-null   float64       
 8   AQHI              1771 non-null   float64       
 9   AQHI_NextDay      1771 non-null   float64       
 10  Season            1771 non-null   str           
 11  Season_Code       1771 non-null   int64         
dtypes: datetime64[us](1), float64(7), int64(3), str(1)
memory usage: 166.2 KB
None


,Date,Day,Month,CO,NO2,O3,PM25,AQHI_PreviousDay,AQHI,AQHI_NextDay,Season,Season_Code
0,2020-01-02,2,1,0.196667,12.708333,22.791667,7.625000,1.886564,2.612882,2.703933,Winter,0
1,2020-01-03,3,1,0.264583,16.041667,16.708333,10.291667,2.612882,2.703933,2.233328,Winter,0
2,2020-01-04,4,1,0.205417,12.291667,19.541667,3.875000,2.703933,2.233328,1.904831,Winter,0
3,2020-01-05,5,1,0.146667,3.250000,29.625000,1.916667,2.233328,1.904831,3.077722,Winter,0
4,2020-01-06,6,1,0.277917,24.041667,10.541667,10.541667,1.904831,3.077722,2.621614,Winter,0


## Step 2: Set Up Features and Target

We drop columns that are not useful for training (Date, Season label, and the target column itself). The target we are trying to predict is AQHI_NextDay, which is the next day AQHI value.

In [3]:
X = data.drop(["index", "Date", "Season", TARGET_VARIABLE], axis=1, errors="ignore")
y = data[TARGET_VARIABLE]

print("Input features:", list(X.columns))
print("Target variable:", TARGET_VARIABLE)
print()
print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

Input features: ['Day', 'Month', 'CO', 'NO2', 'O3', 'PM25', 'AQHI_PreviousDay', 'AQHI', 'Season_Code']
Target variable: AQHI_NextDay

Feature matrix shape: (1771, 9)
Label vector shape: (1771,)


## Step 3: Split into Training and Test Sets

We use a time-based split so that the model is always trained on older data and tested on newer data. This is important for air quality prediction because we want to simulate predicting the future.

- Training set: 2020 to 2023 (80% of data)
- Test set: 2024 (20% of data)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 1416
Test samples: 355


## Step 4: Train the Models

We train five models and print the evaluation metrics for each one after training. The metrics we use are:

- **MSE**: Mean Squared Error, penalizes larger errors more heavily
- **MAE**: Mean Absolute Error, the average size of the error
- **RMSE**: Root Mean Squared Error, same as MSE but in the same units as AQHI
- **MAPE**: Mean Absolute Percentage Error, the average percentage error

In [15]:
# Model 1: Baseline Linear Regression (only uses current AQHI)
base_linear_model = linear_regression_model_base()
base_linear_model.fit(X_train[["AQHI"]], y_train)
y_pred_base = base_linear_model.predict(X_test[["AQHI"]])

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_base)
print("Baseline Linear Regression")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

#Updating the metrics table:
df = pd.read_csv("../results/metrics_table.csv")
df.loc[df["Model"] == "Baseline Linear Regression", ["MSE", "MAE", "RMSE", "MAPE"]] = [mse, mae, rmse, mape]
df.to_csv("../results/metrics_table.csv", index=False)

Baseline Linear Regression
  MSE:  0.1655
  MAE:  0.3127
  RMSE: 0.4068
  MAPE: 0.1417


In [19]:
# Model 2: Linear Regression with all features
linear_model = linear_regression_model_lag()
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_linear)
print("Linear Regression with Lag Features")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

#Updating the metrics table:
df = pd.read_csv("../results/metrics_table.csv")
df.loc[df["Model"] == "Linear Regression with Lag Features", ["MSE", "MAE", "RMSE", "MAPE"]] = [mse, mae, rmse, mape]
df.to_csv("../results/metrics_table.csv", index=False) 

Linear Regression with Lag Features
  MSE:  0.1585
  MAE:  0.3062
  RMSE: 0.3981
  MAPE: 0.1371


In [20]:
# Model 3: Lasso Regression
lasso_model = lasso_regression_model()
param_grid_lasso = get_param_grid_lasso() #Getting the hyperparameter grid for the lasso regression model from the function defined in evaluate.py
grid_search_lasso = get_best_lasso_model(lasso_model, param_grid_lasso, X_train, y_train) #Tuning the lasso regression model for the best hyperparameters using

#Printing the best hyperparameters for the lasso regression model found by GridSearchCV:
print("Best hyperparameters for the lasso regression model found by GridSearchCV:", grid_search_lasso.best_params_)

best_lasso_model_hyperparameters = grid_search_lasso.best_estimator_ #Getting the best model with the best hyperparameters from the grid search results
y_pred_lasso = best_lasso_model_hyperparameters.predict(X_test)

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_lasso)
print("Lasso Regression")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

#Updating the metrics table:
df = pd.read_csv("../results/metrics_table.csv")
df.loc[df["Model"] == "Lasso Regression", ["MSE", "MAE", "RMSE", "MAPE"]] = [mse, mae, rmse, mape]
df.to_csv("../results/metrics_table.csv", index=False)  

Best hyperparameters for the lasso regression model found by GridSearchCV: {'alpha': 0.1}
Lasso Regression
  MSE:  0.1645
  MAE:  0.3117
  RMSE: 0.4055
  MAPE: 0.1400


In [21]:
# Model 4: Decision Tree
DT_model = decision_tree_regression_model()

param_grid_DT = get_param_grid_DT() #Getting the hyperparameter grid for the decision tree regression model from the function defined in evaluate.py
grid_search_DT = get_best_DT_model(DT_model, param_grid_DT, X_train, y_train) #Tuning the decision tree regression model for the best hyperparameters using the function defined in evaluate.py

#Printing the best hyperparameters for the decision tree regression model found by GridSearchCV:
print("Best hyperparameters for the decision tree regression model found by GridSearchCV:", grid_search_DT.best_params_)


best_DT_model_hyperparameters = grid_search_DT.best_estimator_ #Getting the best model with the best hyperparameters from the grid search results
y_pred_DT = best_DT_model_hyperparameters.predict(X_test) #Setting the prediction target to the best estimator found by GridSearchCV

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_DT)
print("Decision Tree")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

#Updating the metrics table:
df = pd.read_csv("../results/metrics_table.csv")
df.loc[df["Model"] == "Decision Tree", ["MSE", "MAE", "RMSE", "MAPE"]] = [mse, mae, rmse, mape]
df.to_csv("../results/metrics_table.csv", index=False)

Best hyperparameters for the decision tree regression model found by GridSearchCV: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10}
Decision Tree
  MSE:  0.2623
  MAE:  0.3957
  RMSE: 0.5122
  MAPE: 0.1766


In [22]:
# Model 5: Random Forest
RF_model = random_forest_regression_model()

param_grid_RF = get_param_grid_RF() #Getting the hyperparameter grid for the random forest regression model from the function defined in evaluate.py
grid_search_RF = get_best_RF_model(RF_model, param_grid_RF, X_train, y_train) #Tuning the random forest regression model for the best hyperparameters using the function defined in evaluate.py

#Printing the best hyperparameters for the random forest regression model found by GridSearchCV:
print("Best hyperparameters for the random forest regression model found by GridSearchCV:", grid_search_RF.best_params_)


best_RF_model_hyperparameters = grid_search_RF.best_estimator_ #Getting the best model with the best hyperparameters from the grid search results
y_pred_RF = best_RF_model_hyperparameters.predict(X_test) #Setting the prediction target to the best estimator found by GridSearchCV

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_RF)
print("Random Forest")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

#Updating the metrics table:
df = pd.read_csv("../results/metrics_table.csv")
df.loc[df["Model"] == "Random Forest", ["MSE", "MAE", "RMSE", "MAPE"]] = [mse, mae, rmse, mape]
df.to_csv("../results/metrics_table.csv", index=False)

Best hyperparameters for the random forest regression model found by GridSearchCV: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}
Random Forest
  MSE:  0.1667
  MAE:  0.3066
  RMSE: 0.4083
  MAPE: 0.1378


## Step 4b: ARIMA (Time Series Model)

The five models above predict `AQHI_NextDay` from the pollutant feature columns. ARIMA is different: it is a pure time series model that forecasts the next AQHI value from the **past AQHI values alone**.

We use the order **(p, d, q) = (2, 0, 0)**, chosen from the ACF/PACF analysis in the EDA notebook:
- **d = 0**: the AQHI series is stationary (rolling mean and standard deviation are stable).
- **p = 2**: the PACF cuts off after about 2 lags.
- **q = 0**: the ACF tails off slowly with no sharp cutoff and no seasonality.

To compare it fairly with the other models, we use a **rolling one-step-ahead** forecast: for each day in the 2024 test set we refit ARIMA on every AQHI value up to that day and forecast the next day, then check it against the same `y_test` the other models use.

In [10]:
# Model 6: ARIMA (pure univariate time series model)
# Unlike the models above, ARIMA does not use the pollutant feature columns.
# It forecasts the next AQHI value directly from the past AQHI values, refitting
# at each step (rolling one-step-ahead) so it is comparable to the other models
# on the same 2024 test days. This refits 355 times, so it takes ~30 seconds.
from evaluate import rolling_arima_forecast
from features import ARIMA_ORDER

split_index = len(X_train)  # same time-based split used above (shuffle=False)
y_pred_arima = rolling_arima_forecast(data["AQHI"].values, split_index, order=ARIMA_ORDER)

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_arima)
print(f"ARIMA {ARIMA_ORDER}")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

ARIMA (2, 0, 0)
  MSE:  0.1645
  MAE:  0.3115
  RMSE: 0.4056
  MAPE: 0.1415


## Step 4b: ARIMA (Time Series Model)

The five models above predict `AQHI_NextDay` from the pollutant feature columns. ARIMA is different: it is a pure time series model that forecasts the next AQHI value from the **past AQHI values alone**.

We use the order **(p, d, q) = (2, 0, 0)**, chosen from the ACF/PACF analysis in the EDA notebook:
- **d = 0**: the AQHI series is stationary (rolling mean and standard deviation are stable).
- **p = 2**: the PACF cuts off after about 2 lags.
- **q = 0**: the ACF tails off slowly with no sharp cutoff and no seasonality.

To compare it fairly with the other models, we use a **rolling one-step-ahead** forecast: for each day in the 2024 test set we refit ARIMA on every AQHI value up to that day and forecast the next day, then check it against the same `y_test` the other models use.

In [10]:
# Model 6: ARIMA (pure univariate time series model)
# Unlike the models above, ARIMA does not use the pollutant feature columns.
# It forecasts the next AQHI value directly from the past AQHI values, refitting
# at each step (rolling one-step-ahead) so it is comparable to the other models
# on the same 2024 test days. This refits 355 times, so it takes ~30 seconds.
from evaluate import rolling_arima_forecast
from features import ARIMA_ORDER

split_index = len(X_train)  # same time-based split used above (shuffle=False)
y_pred_arima = rolling_arima_forecast(data["AQHI"].values, split_index, order=ARIMA_ORDER)

mse, mae, rmse, mape = model_evaluation(y_test, y_pred_arima)
print(f"ARIMA {ARIMA_ORDER}")
print(f"  MSE:  {mse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAPE: {mape:.4f}")

ARIMA (2, 0, 0)
  MSE:  0.1645
  MAE:  0.3115
  RMSE: 0.4056
  MAPE: 0.1415


## Step 5: Results Summary

Here is a side by side comparison of all six models. The table is recomputed from each model's predictions every time this notebook runs, so it always reflects the current code, and the values are saved to `../results/metrics_table.csv`.

In [11]:
# Build the results table from the latest predictions so it always reflects
# the current code. We recompute each model's metrics from its stored
# predictions (y_pred_*), write them to the CSV, then display the table.
import os

model_predictions = {
    "Baseline Linear Regression": y_pred_base,
    "Linear Regression with Lag Features": y_pred_linear,
    "Lasso Regression": y_pred_lasso,
    "Decision Tree": y_pred_DT,
    "Random Forest": y_pred_RF,
    "ARIMA": y_pred_arima,
}

rows = []
for name, y_pred in model_predictions.items():
    mse, mae, rmse, mape = model_evaluation(y_test, y_pred)
    rows.append({"Model": name, "MSE": mse, "MAE": mae, "RMSE": rmse, "MAPE": mape})

results = pd.DataFrame(rows)

# Save the freshly computed metrics so the CSV stays in sync with the code
os.makedirs("../results", exist_ok=True)
results.to_csv("../results/metrics_table.csv", index=False)

results

,Model,MSE,MAE,RMSE,MAPE
0,Baseline Linear Regression,0.165493,0.312738,0.406808,0.141705
1,Linear Regression with Lag Features,0.158522,0.306216,0.398148,0.137058
2,Lasso Regression,0.164469,0.311664,0.405548,0.139952
3,Decision Tree,0.180569,0.322747,0.424934,0.146006
4,Random Forest,0.164510,0.303917,0.405598,0.136977
5,ARIMA,0.164507,0.311456,0.405595,0.141521
